<a href="https://colab.research.google.com/github/sevaradevoloper/machine-learning-class-work/blob/main/CatBoost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [94]:
import opendatasets as od

In [95]:
od.download("https://www.kaggle.com/datasets/kanyianalyst/car-evaluation-dataset/data")

Skipping, found downloaded files in "./car-evaluation-dataset" (use force=True to force download)


In [96]:
import pandas as pd
df = pd.read_csv("/content/car-evaluation-dataset/car evaluation_with.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1726 entries, 0 to 1725
Data columns (total 7 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   vhigh    1726 non-null   object
 1   vhigh.1  1726 non-null   object
 2   2        1726 non-null   int64 
 3   2.1      1726 non-null   int64 
 4   small    1726 non-null   object
 5   med      1726 non-null   object
 6   unacc    1726 non-null   object
dtypes: int64(2), object(5)
memory usage: 94.5+ KB


In [97]:
print(df.columns.tolist())

['vhigh', 'vhigh.1', '2', '2.1', 'small', 'med', 'unacc']


In [98]:
df

,vhigh,vhigh.1,2,2.1,small,med,unacc
0,vhigh,vhigh,2,2,small,high,unacc
1,vhigh,vhigh,2,2,med,low,unacc
2,vhigh,vhigh,2,2,med,med,unacc
3,vhigh,vhigh,2,2,med,high,unacc
4,vhigh,vhigh,2,2,big,low,unacc
...,...,...,...,...,...,...,...
1721,low,low,5,5,med,med,good
1722,low,low,5,5,med,high,vgood
1723,low,low,5,5,big,low,unacc
1724,low,low,5,5,big,med,good


In [99]:
# 1. Ustun nomlarini o'zbekcha so'zlarga o'zgartiramiz
df.columns = ['sotib_olish_narxi', 'remont_xarajati', 'eshiklar_soni', 'odam_sigimi', 'bagaj_hajmi', 'xavfsizlik', 'mashina_bahosi']

# 2. To'g'rilangan jadvalning birinchi 5 ta qatorini ko'ramiz
df.head()


,sotib_olish_narxi,remont_xarajati,eshiklar_soni,odam_sigimi,bagaj_hajmi,xavfsizlik,mashina_bahosi
0,vhigh,vhigh,2,2,small,high,unacc
1,vhigh,vhigh,2,2,med,low,unacc
2,vhigh,vhigh,2,2,med,med,unacc
3,vhigh,vhigh,2,2,med,high,unacc
4,vhigh,vhigh,2,2,big,low,unacc


In [100]:
df.isnull().sum()
df.duplicated().sum()

np.int64(0)

In [101]:
for ustun in df.columns:
  print(f"{ustun} ustuni qiymatlar: {df[ustun].unique()}")
  print("===="*20)

sotib_olish_narxi ustuni qiymatlar: ['vhigh' 'high' 'med' 'low']
remont_xarajati ustuni qiymatlar: ['vhigh' 'high' 'med' 'low']
eshiklar_soni ustuni qiymatlar: [2 3 4 5]
odam_sigimi ustuni qiymatlar: [2 4 5]
bagaj_hajmi ustuni qiymatlar: ['small' 'med' 'big']
xavfsizlik ustuni qiymatlar: ['high' 'low' 'med']
mashina_bahosi ustuni qiymatlar: ['unacc' 'acc' 'vgood' 'good']


In [102]:
print(df['mashina_bahosi'].value_counts())
# target ustunimizani ichida qaysi qiymat nechi martadan uchrashini tekshirib olamiz
# imbalanced dataset ekan

mashina_bahosi
unacc    1208
acc       384
good       69
vgood      65
Name: count, dtype: int64


In [103]:
# x va y ajratamiz
from sklearn.model_selection import train_test_split

In [104]:
X = df.drop(columns = "mashina_bahosi",axis = 1)
y = df["mashina_bahosi"]
# olchamlarini tekshirib olamiz

print(X.shape)
print(y.shape)

(1726, 6)
(1726,)


In [105]:
X_train,X_test,y_train,y_test = train_test_split(X,y,random_state=42,test_size=0.2,stratify=y)
print("X_train o'lchami:", X_train.shape)
print("X_test o'lchami:", X_test.shape)
print("y_train o'lchami:", y_train.shape)
print("y_test o'lchami:", y_test.shape)

X_train o'lchami: (1380, 6)
X_test o'lchami: (346, 6)
y_train o'lchami: (1380,)
y_test o'lchami: (346,)


In [106]:
# cat_features ro'yxatini shakllantirish
df.columns.tolist()

cat_feature = ['sotib_olish_narxi',
 'remont_xarajati',
 'eshiklar_soni',
 'odam_sigimi',
 'bagaj_hajmi',
 'xavfsizlik',]

print(f"categorik ustunlar {cat_feature}")
# yoki


# # object (matn) turidagi barcha ustunlar royxatini olamiz
# cat_features = X_train.select_dtypes(include='object').columns.tolist()

# print("Kategoriyali ustunlar:", cat_features)

categorik ustunlar ['sotib_olish_narxi', 'remont_xarajati', 'eshiklar_soni', 'odam_sigimi', 'bagaj_hajmi', 'xavfsizlik']


In [107]:
# Model parametrlarini sozlash va fit qilish
from catboost import CatBoostClassifier
import catboost as cb


cat_model = CatBoostClassifier(
    iterations=500,              # nechta daraxt qurishi
    learning_rate=0.1,           # har bir qadamda qanchalik "o'rganishi"
    loss_function='MultiClass',  # 4 ta sinf bo'lgani uchun ko'p-sinfli
    random_seed=42,              # natija takrorlanadigan bo'lishi uchun
    verbose=100                  # har 100 qadamda hisobot chiqarsin
)

In [108]:
cat_model.fit(
    X_train,y_train,
    cat_features = cat_feature,
    eval_set=(X_test, y_test)
)

0:	learn: 1.2532327	test: 1.2450430	best: 1.2450430 (0)	total: 13.6ms	remaining: 6.81s
100:	learn: 0.1447587	test: 0.1165576	best: 0.1165576 (100)	total: 1.65s	remaining: 6.52s
200:	learn: 0.0946614	test: 0.0894782	best: 0.0894782 (200)	total: 3.1s	remaining: 4.61s
300:	learn: 0.0666033	test: 0.0712159	best: 0.0712159 (300)	total: 4.63s	remaining: 3.06s
400:	learn: 0.0481666	test: 0.0609908	best: 0.0609908 (400)	total: 6.11s	remaining: 1.51s
499:	learn: 0.0370436	test: 0.0549810	best: 0.0549369 (498)	total: 7.56s	remaining: 0us

bestTest = 0.05493690119
bestIteration = 498

Shrink model to first 499 iterations.


CatBoostClassifier(iterations=500, learning_rate=0.1, loss_function='MultiClass', random_seed=42, verbose=100)

In [109]:
# model predict
cat_pred = cat_model.predict(X_test)
print(cat_pred[:10])

[['good']
 ['unacc']
 ['unacc']
 ['unacc']
 ['acc']
 ['unacc']
 ['unacc']
 ['acc']
 ['unacc']
 ['unacc']]


In [110]:
# accurancy
from sklearn.metrics import classification_report,accuracy_score,confusion_matrix

accuracy = accuracy_score(y_test,cat_pred)
print(f"Model aniqligi (Accuracy): {accuracy:.4f}")
print(f"Foizda: {accuracy * 100:.2f}%")


cat_report = classification_report(y_test,cat_pred)
print("===" * 20)
print(f"Catboost report: {cat_report}")

Model aniqligi (Accuracy): 0.9798
Foizda: 97.98%
Catboost report:               precision    recall  f1-score   support

         acc       0.96      0.96      0.96        77
        good       0.88      1.00      0.93        14
       unacc       1.00      0.98      0.99       242
       vgood       0.93      1.00      0.96        13

    accuracy                           0.98       346
   macro avg       0.94      0.99      0.96       346
weighted avg       0.98      0.98      0.98       346



Butun loyiha xulosasi (yodda saqlash uchun) 🎓

Siz to'liq bir ML pipeline'ni bosqichma-bosqich qurib chiqdingiz:

Bosqich	Nima qildik	Asosiy tushuncha
0. EDA	Ma'lumotni tanidik	Hamma ustun matnli, target balanssiz
1. X/y	Belgi va targetni ajratdik	drop(axis=1)
2. Split	Train/Test'ga bo'ldik	CatBoost'da Encoding/Scaling shart emas + stratify
3. cat_features	Kategoriyali ustunlarni belgiladik	select_dtypes('object'), target'siz!
4. Fit	Modelni o'qitdik	loss_function='MultiClass', cat_features
5. Baholash	Natijani sinadik	Accuracy + classification_report
Eng katta saboq: CatBoost'ning kuchi — matnli (kategoriyali) ma'lumot bilan qo'lda Encoding/Scaling'siz ishlay olishida. Faqat unga cat_features ni to'g'ri ko'rsatish kifoya. 🌟

In [111]:
# Boshqa modellar bn solishtirish uchun Encoding qilib chiqamiz:

In [112]:
from sklearn.preprocessing import OrdinalEncoder
oridinal_encoder = OrdinalEncoder()

X_train_enc = oridinal_encoder.fit_transform(X_train)
X_test_enc = oridinal_encoder.transform(X_test)

from sklearn.preprocessing import OrdinalEncoder

# Natijani tekshirish
print("Encoding'dan oldin (matn):")
print(X_train.head(3))
print("\nEncoding'dan keyin (raqam):")
print(X_train_enc[:3])


Encoding'dan oldin (matn):
     sotib_olish_narxi remont_xarajati  eshiklar_soni  odam_sigimi  \
1118               med             med              3            4   
227              vhigh             med              2            4   
1502               low            high              5            5   

     bagaj_hajmi xavfsizlik  
1118         med        med  
227          med        med  
1502       small        med  

Encoding'dan keyin (raqam):
[[2. 2. 1. 1. 1. 2.]
 [3. 2. 0. 1. 1. 2.]
 [1. 0. 3. 2. 2. 2.]]


In [113]:
X_train_enc = pd.DataFrame(
    oridinal_encoder.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test_enc = pd.DataFrame(
    oridinal_encoder.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

# Теперь эта строчка отработает идеально и без ошибок:
print(X_train_enc['xavfsizlik'].head(3))
print(X_train_enc)

1118    2.0
227     2.0
1502    2.0
Name: xavfsizlik, dtype: float64
      sotib_olish_narxi  remont_xarajati  eshiklar_soni  odam_sigimi  \
1118                2.0              2.0            1.0          1.0   
227                 3.0              2.0            0.0          1.0   
1502                1.0              0.0            3.0          2.0   
1721                1.0              1.0            3.0          2.0   
973                 2.0              0.0            0.0          0.0   
...                 ...              ...            ...          ...   
83                  3.0              3.0            3.0          0.0   
212                 3.0              0.0            3.0          2.0   
1303                1.0              3.0            0.0          1.0   
313                 3.0              2.0            3.0          2.0   
1253                2.0              1.0            2.0          1.0   

      bagaj_hajmi  xavfsizlik  
1118          1.0         2.0  
22

In [114]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.preprocessing import LabelEncoder

# XGBoost target'ni 0,1,2,3 ko'rinishida kutadi — shuning uchun y ni ham encode qilamiz
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

# Modellar lug'ati
modellar = {
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42),
    "XGBoost":      XGBClassifier(n_estimators=200, learning_rate=0.1, random_state=42, verbosity=0),
    "LightGBM":     LGBMClassifier(n_estimators=200, learning_rate=0.1, random_state=42, verbose=-1),
}

In [115]:
from sklearn.metrics import accuracy_score
import time
import pandas as pd

natijalar = []  # natijalarni yig'ish uchun ro'yxat

for nom, model in modellar.items():
    start = time.perf_counter()          # vaqtni o'lchash boshlanishi

    model.fit(X_train_enc, y_train_enc)  # raqamli ma'lumotda o'qitish
    y_pred = model.predict(X_test_enc)   # bashorat

    davomiylik = time.perf_counter() - start
    acc = accuracy_score(y_test_enc, y_pred)
    class_report = classification_report(y_test_enc,y_pred)

    natijalar.append({
        "Model": nom,
        "Accuracy": round(acc, 4),
        "Classifier report":class_report,
        "Vaqt (s)": round(davomiylik, 3)
    })
    print(f"✅ {nom} tayyor — Accuracy: {acc:.4f}")
    print(f"✅ {nom} tayyor — Classifier report: {class_report}")

✅ RandomForest tayyor — Accuracy: 0.9798
✅ RandomForest tayyor — Classifier report:               precision    recall  f1-score   support

           0       0.95      0.96      0.95        77
           1       0.93      0.93      0.93        14
           2       0.99      0.99      0.99       242
           3       1.00      0.92      0.96        13

    accuracy                           0.98       346
   macro avg       0.97      0.95      0.96       346
weighted avg       0.98      0.98      0.98       346

✅ XGBoost tayyor — Accuracy: 0.9913
✅ XGBoost tayyor — Classifier report:               precision    recall  f1-score   support

           0       1.00      0.97      0.99        77
           1       0.88      1.00      0.93        14
           2       1.00      1.00      1.00       242
           3       1.00      0.92      0.96        13

    accuracy                           0.99       346
   macro avg       0.97      0.97      0.97       346
weighted avg       0.99    

In [116]:
# CatBoost natijasini qo'lda qo'shamiz (matnli ma'lumotda ishlagani uchun)
natijalar.append({
    "Model": "CatBoost",
    "Accuracy": 0.9798,                    # 5-bosqichda olgan natijangiz
    "Vaqt (s)": 0.542,                     # Agar o'lchagan bo'lsangiz soniyani yozing (masalan, 0.542), aks holda None
    "Classifier report": "CatBoost report" # Bu yerga hisobot matnini qo'yishingiz mumkin
})


In [117]:
# DataFrame yaratamiz va Accuracy bo'yicha tartiblaymiz
df_results = pd.DataFrame(natijalar)
df_results = df_results.sort_values(by="Accuracy", ascending=False).reset_index(drop=True)

print("\n📊 MODELLAR SOLISHTIRUVI:")
df_results


📊 MODELLAR SOLISHTIRUVI:


,Model,Accuracy,Classifier report,Vaqt (s)
0,LightGBM,1.0000,precision recall f1-score ...,0.245
1,XGBoost,0.9913,precision recall f1-score ...,0.190
2,RandomForest,0.9798,precision recall f1-score ...,0.298
3,CatBoost,0.9798,CatBoost report,0.542
